# Sanskrit & Indic NLP Tooling Testbed

A test notebook to integrate Sanskrit & Indic NLP tooling (SHP, transliteration, IPA) into nyaya-quantum-nlp and enrich `gui_verses.jsonl`.


In [15]:
# 1) Setup Paths, Config Flags, and Caches
from pathlib import Path
import json, time, datetime, shutil, sys

ROOT = Path(r"c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp")
DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CACHE_DIR = DATA_DIR / "cache"
for d in (PROCESSED_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Feature flags
USE_SHP = True
USE_AKSHARAMUKHA = True
USE_PHONEMIZER = False  # will auto-disable if deps missing

# Minimal JSONL IO helpers (reuse if already defined elsewhere)
def read_jsonl(path: Path):
    if not path.exists():
        return []
    out = []
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    out.append(obj)
            except Exception:
                continue
    return out


def write_jsonl(path: Path, records):
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


class Timer:
    def __init__(self, label=""): self.label = label; self.t0 = None
    def __enter__(self): self.t0 = time.time(); return self
    def __exit__(self, exc_type, exc, tb):
        dt = time.time() - self.t0
        print(f"[time] {self.label} took {dt:.3f}s")
        self.dt = dt


In [16]:
# 2) Install/Verify External Libraries
print("[deps] verifying libraries...")

def ensure(package, import_name=None):
    try:
        return __import__(import_name or package)
    except Exception:
        import subprocess
        print(f"[pip] installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        return __import__(import_name or package)

# Core
requests = ensure("requests")
bs4 = ensure("beautifulsoup4", "bs4")

# Transliteration libs
aksha = None
if USE_AKSHARAMUKHA:
    try:
        aksha = ensure("aksharamukha")
    except Exception:
        aksha = None

sanscript = None
try:
    sanskrit_pkg = ensure("indic_transliteration")
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import SchemeMap, SCHEMES, transliterate as _itr
    sanscript = _sanscript
except Exception:
    sanscript = None

# CLTK (optional)
cltk = None
try:
    cltk = ensure("cltk")
except Exception:
    cltk = None

# Phonemizer (optional)
phonemizer = None
if USE_PHONEMIZER:
    try:
        phonemizer = ensure("phonemizer")
        # crude espeak check: phonemizer uses espeak-ng backend typically
        import shutil as _sh
        if not _sh.which("espeak-ng"):
            print("[phonemizer] espeak-ng not found; disabling phonemizer")
            phonemizer = None
            USE_PHONEMIZER = False
    except Exception:
        phonemizer = None
        USE_PHONEMIZER = False

print("[deps] ready:")
print({
    "requests": getattr(requests, "__version__", "ok"),
    "bs4": getattr(bs4, "__version__", "ok"),
    "aksharamukha": bool(aksha),
    "indic_transliteration": bool(sanscript),
    "cltk": bool(cltk),
    "phonemizer": bool(phonemizer),
})


[deps] verifying libraries...
[deps] ready:
{'requests': '2.32.3', 'bs4': '4.12.3', 'aksharamukha': True, 'indic_transliteration': True, 'cltk': True, 'phonemizer': False}


In [17]:
# 3) Load Sample Verses (gui_verses.jsonl)
SRC = PROCESSED_DIR / "gui_verses.jsonl"
records = read_jsonl(SRC)
if not records:
    print("[data] gui_verses.jsonl not found; synthesizing small sample")
    records = [
        {"id": "1.1.1", "book": "1", "sarga": "1", "sa_verse": "ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वरम् ॥", "en": "..."},
        {"id": "1.1.2", "book": "1", "sarga": "1", "sa_verse": "कोन्वस्मिन् साम्प्रतं लोके गुणवान् कश्च वीर्यवान् ।", "en": "..."},
    ]
# Keep minimal fields
for r in records:
    r["id"] = str(r.get("id", "1.1.1"))
    r["book"] = str(r.get("book", "1"))
    r["sarga"] = str(r.get("sarga", "1"))
    r["sa_verse"] = r.get("sa_verse", "").strip()
    r["en"] = r.get("en", "").strip()

print(f"[data] loaded {len(records)} records")
print(json.dumps(records[:2], ensure_ascii=False, indent=2))


[data] loaded 19182 records
[
  {
    "id": "1.1.1",
    "book": "1",
    "sarga": "1",
    "sa_verse": "ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वरम्। नारदं परिपप्रच्छ वाल्मीकिर्मुनिपुङ्गवम्॥",
    "en": "The ascetic Vālmīki asked Nārada, the best of sages and foremost of those conversant with words, ever engaged in austerities and Vedic studies.",
    "source": "scrape_rahular",
    "version": "1.0"
  },
  {
    "id": "1.1.2",
    "book": "1",
    "sarga": "1",
    "sa_verse": "कोन्वस्मिन् साम्प्रतं लोके गुणवान् कश्च वीर्यवान्। धर्मज्ञश्च कृतज्ञश्च सत्यवाक्यो दृढत्नतः॥",
    "en": "Who at present in this world is like crowned with qualities, and with prowess, knowing duty, and grateful, and truthful, and firm in vow.",
    "source": "scrape_rahular",
    "version": "1.0"
  }
]


In [18]:
# 4) Transliteration Helpers (Aksharamukha / Indic-Transliteration)

def dev_to_iast(text: str) -> str:
    if not text:
        return ""
    # Try Aksharamukha
    if aksha is not None:
        try:
            from aksharamukha import transliterate
            return transliterate.process("Devanagari", "IAST", text)
        except Exception:
            pass
    # Fallback to indic_transliteration
    if sanscript is not None:
        try:
            return _itr(text, sanscript.DEVANAGARI, sanscript.IAST)
        except Exception:
            pass
    return text


def iast_to_dev(text: str) -> str:
    if not text:
        return ""
    if aksha is not None:
        try:
            from aksharamukha import transliterate
            return transliterate.process("IAST", "Devanagari", text)
        except Exception:
            pass
    if sanscript is not None:
        try:
            return _itr(text, sanscript.IAST, sanscript.DEVANAGARI)
        except Exception:
            pass
    return text


def dev_to_slp1(text: str) -> str:
    if not text:
        return ""
    if sanscript is not None:
        try:
            return _itr(text, sanscript.DEVANAGARI, sanscript.SLP1)
        except Exception:
            pass
    if aksha is not None:
        try:
            from aksharamukha import transliterate
            # Aksharamukha supports SLP1 as "SLP1"
            return transliterate.process("Devanagari", "SLP1", text)
        except Exception:
            pass
    return text


def batch_transliterate(texts, fn=dev_to_iast):
    out = []
    for t in texts:
        try:
            out.append(fn(t))
        except Exception:
            out.append(t)
    return out

print("[trl] demo:")
print(dev_to_iast(records[0]["sa_verse"]))


[trl] demo:
oṃ tapaḥ svādhyāyanirataṃ tapasvī vāgvidāṃ varam. nāradaṃ paripapraccha vālmīkirmunipuṅgavam..


In [19]:
# 5) Sanskrit Tokenization Utilities
import re
DANDA = "\u0964"
DOUBLE_DANDA = "\u0965"

DEV_RANGE = (0x0900, 0x097F)

def is_devanagari(ch: str) -> bool:
    return len(ch) == 1 and DEV_RANGE[0] <= ord(ch) <= DEV_RANGE[1]


def normalize_sanskrit(text: str) -> str:
    s = (text or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def split_sanskrit(text: str):
    """Basic Sanskrit tokenization preserving dandas; splits on spaces and punctuation."""
    s = normalize_sanskrit(text)
    # keep dandas as separate tokens
    s = s.replace(DOUBLE_DANDA, f" {DOUBLE_DANDA} ").replace(DANDA, f" {DANDA} ")
    # split on non-letter/digit but keep Devanagari
    tokens = []
    buff = []
    for ch in s:
        if ch.isspace():
            if buff:
                tokens.append("".join(buff)); buff = []
        elif ch in {DANDA, DOUBLE_DANDA}:
            if buff:
                tokens.append("".join(buff)); buff = []
            tokens.append(ch)
        elif ch.isalnum() or is_devanagari(ch):
            buff.append(ch)
        else:
            if buff:
                tokens.append("".join(buff)); buff = []
    if buff:
        tokens.append("".join(buff))
    # drop pure danda tokens
    return [t for t in tokens if t not in {DANDA, DOUBLE_DANDA}]


def dedup_tokens(tokens):
    seen, out = set(), []
    for t in tokens:
        if t not in seen:
            seen.add(t); out.append(t)
    return out

print("[tok] demo:", split_sanskrit(records[0]["sa_verse"])[:10])


[tok] demo: ['ॐ', 'तपः', 'स्वाध्यायनिरतं', 'तपस्वी', 'वाग्विदां', 'वरम्', 'नारदं', 'परिपप्रच्छ', 'वाल्मीकिर्मुनिपुङ्गवम्']


In [20]:
# 6) Sanskrit Heritage Platform Client (HTTP + Cache)
import hashlib
from pathlib import Path
from typing import List, Dict

SHP_CACHE_PATH = CACHE_DIR / "shp_morph_cache.json"
# Primary analyzer endpoint (HTML). We'll query using 'text' param (preferred) and fall back to 't'.
SHP_BASE = "https://sanskrit.inria.fr/cgi-bin/SKT/sktgraph"
SHP_ANALYZE = "https://sanskrit.inria.fr/cgi-bin/SKT/sktgraph?lex=SH&c=g&ch=s&topic=vyak&mode=g&auth=SH&noR=2"
SHP_HEADERS = {
    "User-Agent": "nyaya-quantum-nlp/0.1 (+local-notebook)",
    "Accept-Language": "en",
}
VERBOSE_SHP = False  # reduce noisy errors

# Cache helpers
try:
    _shp_cache = json.loads(SHP_CACHE_PATH.read_text(encoding="utf-8")) if SHP_CACHE_PATH.exists() else {}
except Exception:
    _shp_cache = {}


def _cache_key(token_norm: str) -> str:
    return hashlib.sha1(token_norm.encode("utf-8")).hexdigest()


def save_shp_cache():
    try:
        SHP_CACHE_PATH.write_text(json.dumps(_shp_cache, ensure_ascii=False, indent=0), encoding="utf-8")
    except Exception as e:
        if VERBOSE_SHP:
            print(f"[cache] save failed: {e}")


def _token_is_devanagari_word(tok: str) -> bool:
    if not tok:
        return False
    has_dev = any(is_devanagari(ch) for ch in tok)
    # skip if token is just punctuation/marks
    alnum_or_dev = any(ch.isalnum() or is_devanagari(ch) for ch in tok)
    return has_dev and alnum_or_dev


def _parse_shp_html_for_analyses(html: str, token_hint: str) -> List[Dict]:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    analyses: List[Dict] = []

    # Heuristic: scan table rows and list items; collect meaningful lines
    def add(line: str):
        line = (line or "").strip()
        if not line:
            return
        # crude lemma extraction: take first token-like chunk
        lemma = line.split("=")[0].strip().split(" ")[0]
        analyses.append({
            "token": token_hint,
            "lemma": lemma,
            "pos": "",
            "feats": "",
            "raw": line,
        })

    for tr in soup.select("table tr"):
        cells = [c.get_text(" ", strip=True) for c in tr.find_all(["td", "th"])]
        line = " | ".join([c for c in cells if c])
        if line and any(k in line.lower() for k in ["lemma", "pos", "cat", "case", "gender", "number", "morph"]):
            add(line)
    if not analyses:
        for li in soup.select("li"):
            txt = li.get_text(" ", strip=True)
            # keep somewhat informative lines
            if len(txt) > 2 and any(ch.isalpha() for ch in txt):
                add(txt)
    return analyses


def shp_analyze_token(token_dev: str) -> List[Dict]:
    """Query SHP for a single token, return list of analyses [{lemma,pos,feats,raw}].
    - Transliterates Devanagari to IAST for robustness.
    - Tries multiple query styles and parses HTML heuristically.
    - Caches results and rate-limits.
    """
    token = (token_dev or "").strip()
    if not token:
        return []
    if token == "ॐ" or not _token_is_devanagari_word(token):
        return []

    # Normalize/cache by IAST to avoid duplicates across scripts
    try:
        token_iast = dev_to_iast(token)
    except Exception:
        token_iast = token
    key = _cache_key(f"IAST::{token_iast}")
    if key in _shp_cache:
        return _shp_cache[key]
    if not USE_SHP:
        return []

    analyses: List[Dict] = []
    last_err = None
    with Timer(f"SHP {token}"):
        try_variants = [
            # Preferred: 'text' parameter
            (SHP_BASE, {"lex": "SH", "text": token_iast, "topic": "vyak", "mode": "g", "noR": "2"}),
            # Fallback: 't' parameter
            (SHP_BASE, {"lex": "SH", "t": token_iast, "topic": "vyak", "mode": "g", "noR": "2"}),
            # Prebuilt analyzer URL with params
            (SHP_ANALYZE, {"text": token_iast}),
        ]
        for url, params in try_variants:
            try:
                r = requests.get(url, params=params, timeout=25, headers=SHP_HEADERS)
                if r.status_code != 200:
                    last_err = f"HTTP {r.status_code}"
                    continue
                html = r.text or ""
                analyses = _parse_shp_html_for_analyses(html, token_hint=token)
                if analyses:
                    break
            except Exception as e:
                last_err = str(e)
                continue
    if not analyses and VERBOSE_SHP and last_err:
        print(f"[shp] warn '{token}': {last_err}")

    _shp_cache[key] = analyses
    save_shp_cache()
    time.sleep(0.25)  # rate-limit
    return analyses

print("[shp] client ready; cache entries:", len(_shp_cache))

[shp] client ready; cache entries: 469


In [21]:
# 7) Morphological Analysis Pipeline for Sanskrit Verses
from typing import Dict, Any


def best_analysis(analyses):
    if not analyses:
        return {"lemma": "", "pos": "", "feats": "", "raw": ""}
    return analyses[0]


_SKIP_SET = {"ॐ", "।", "॥"}


def _is_meaningful_token(t: str) -> bool:
    if not t or t in _SKIP_SET:
        return False
    # drop very short tokens which often are particles/punct
    if len(t) == 1 and not t.isalpha() and not is_devanagari(t):
        return False
    return True


def analyze_verse(rec: Dict[str, Any]):
    sa = rec.get("sa_verse", "")
    toks = split_sanskrit(sa)
    toks = [t for t in toks if _is_meaningful_token(t.strip())]
    morph = []
    for t in toks:
        analyses = shp_analyze_token(t)
        best = best_analysis(analyses)
        morph.append({
            "token": t,
            "lemma": best.get("lemma", ""),
            "pos": best.get("pos", ""),
            "feats": best.get("feats", ""),
            "analysis_raw": best.get("raw", ""),
        })
    out = dict(rec)
    out["morph"] = morph
    return out

print("[morph] demo on first record...")
_demo = analyze_verse(records[0])
print(json.dumps({"id": _demo["id"], "morph_len": len(_demo.get("morph", []))}, ensure_ascii=False))

[morph] demo on first record...
[time] SHP तपः took 2.297s
[time] SHP तपः took 2.297s
[time] SHP स्वाध्यायनिरतं took 1.698s
[time] SHP स्वाध्यायनिरतं took 1.698s
[time] SHP तपस्वी took 1.758s
[time] SHP तपस्वी took 1.758s
[time] SHP वाग्विदां took 1.748s
[time] SHP वाग्विदां took 1.748s
[time] SHP वरम् took 1.869s
[time] SHP वरम् took 1.869s
[time] SHP नारदं took 2.039s
[time] SHP नारदं took 2.039s
[time] SHP परिपप्रच्छ took 2.301s
[time] SHP परिपप्रच्छ took 2.301s
[time] SHP वाल्मीकिर्मुनिपुङ्गवम् took 2.330s
[time] SHP वाल्मीकिर्मुनिपुङ्गवम् took 2.330s
{"id": "1.1.1", "morph_len": 8}
{"id": "1.1.1", "morph_len": 8}


In [22]:
# 8) IPA Generation (Aksharamukha and Phonemizer Fallback)

def to_ipa_dev(text_dev: str) -> str:
    s = (text_dev or "").strip()
    if not s:
        return ""
    # Try Aksharamukha target IPA
    if aksha is not None:
        try:
            from aksharamukha import transliterate
            return transliterate.process("Devanagari", "IPA", s)
        except Exception:
            pass
    # Fallback: phonemizer via Hindi if available
    if phonemizer is not None and USE_PHONEMIZER:
        try:
            from phonemizer import phonemize
            return phonemize(s, language='hi', backend='espeak', strip=True)
        except Exception:
            pass
    return ""


def ipa_tokens(tokens):
    return [to_ipa_dev(t) for t in tokens]

print("[ipa] demo:", to_ipa_dev(records[0]["sa_verse"])[:60], "...")


[ipa] demo: oːm t̪əpəhə̆ s̪ʋɑːd̪ʰjɑːjən̪ɪɾət̪ə̃ t̪əpəs̪ʋiː ʋɑːgʋɪd̪ɑ̃ː ʋ ...


In [23]:
# 9) Enrich Records and Incremental Write (gui_verses_enriched.jsonl)
ENRICH_TARGET = PROCESSED_DIR / "gui_verses_enriched.jsonl"
BACKUP_DIR = PROCESSED_DIR / "backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)


def backup_file(path: Path):
    if path.exists():
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        bak = BACKUP_DIR / f"{path.stem}_{ts}{path.suffix}"
        shutil.copy2(path, bak)
        print(f"[backup] {bak}")


def enrich_record(rec):
    out = dict(rec)
    sa = rec.get("sa_verse", "")
    toks = split_sanskrit(sa)
    out["iast"] = dev_to_iast(sa)
    out_m = analyze_verse(rec)
    out["morph"] = out_m.get("morph", [])
    out["ipa"] = to_ipa_dev(sa)
    tags = list(out.get("source_tags", []))
    if "enriched" not in tags:
        tags.append("enriched")
    out["source_tags"] = tags
    return out

# Process N records with progress and incremental writes
N = min(50, len(records))  # cap for testbed; adjust as needed
print(f"[enrich] processing {N} records...")
acc = []
backup_file(ENRICH_TARGET)
with Timer("enrich batch"):
    for i, rec in enumerate(records[:N], start=1):
        acc.append(enrich_record(rec))
        if i % 5 == 0 or i == N:
            write_jsonl(ENRICH_TARGET, acc)
            print(f"[enrich] wrote {i}/{N}")
print(f"[enrich] done -> {ENRICH_TARGET}")


[enrich] processing 50 records...
[backup] c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\backups\gui_verses_enriched_20250817_214427.jsonl
[time] SHP कोन्वस्मिन् took 2.225s
[time] SHP कोन्वस्मिन् took 2.225s
[time] SHP साम्प्रतं took 2.193s
[time] SHP साम्प्रतं took 2.193s
[time] SHP लोके took 2.664s
[time] SHP लोके took 2.664s
[time] SHP गुणवान् took 1.750s
[time] SHP गुणवान् took 1.750s
[time] SHP कश्च took 1.874s
[time] SHP कश्च took 1.874s
[time] SHP वीर्यवान् took 2.221s
[time] SHP वीर्यवान् took 2.221s
[time] SHP धर्मज्ञश्च took 1.895s
[time] SHP धर्मज्ञश्च took 1.895s
[time] SHP कृतज्ञश्च took 2.622s
[time] SHP कृतज्ञश्च took 2.622s
[time] SHP सत्यवाक्यो took 2.035s
[time] SHP सत्यवाक्यो took 2.035s
[time] SHP दृढत्नतः took 2.365s
[time] SHP दृढत्नतः took 2.365s
[time] SHP चारित्रेण took 1.966s
[time] SHP चारित्रेण took 1.966s
[time] SHP च took 1.757s
[time] SHP च took 1.757s
[time] SHP को took 1.850s
[time] SHP को took 1.850s
[time] SHP युक्तः took 2.047s
[t

In [24]:
# 10) Quality Report: Coverage, Unknown Tokens, Timing
from collections import Counter

enriched = read_jsonl(ENRICH_TARGET)
print(f"[report] loaded {len(enriched)} enriched records")

total_tokens = 0
unknown = Counter()
per_verse_cov = []

for rec in enriched:
    morph = rec.get("morph", [])
    toks = [m.get("token", "") for m in morph if m.get("token")]
    total_tokens += len(toks)
    unk = sum(1 for m in morph if not m.get("lemma"))
    if toks:
        per_verse_cov.append(1 - unk/len(toks))
    for m in morph:
        if not m.get("lemma") and m.get("token"):
            unknown[m["token"]] += 1

summary = {
    "records": len(enriched),
    "total_tokens": total_tokens,
    "avg_coverage": round(sum(per_verse_cov)/max(1, len(per_verse_cov)), 3) if per_verse_cov else 0.0,
    "top_unknown": unknown.most_common(10),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

REPORT_PATH = PROCESSED_DIR / "enrich_report.json"
REPORT_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"[report] saved -> {REPORT_PATH}")


[report] loaded 50 enriched records
{
  "records": 50,
  "total_tokens": 581,
  "avg_coverage": 0.0,
  "top_unknown": [
    [
      "च",
      15
    ],
    [
      "स",
      9
    ],
    [
      "तु",
      6
    ],
    [
      "रामो",
      5
    ],
    [
      "रामं",
      5
    ],
    [
      "ह",
      5
    ],
    [
      "वने",
      5
    ],
    [
      "श्रुत्वा",
      4
    ],
    [
      "जगाम",
      4
    ],
    [
      "रामस्य",
      4
    ]
  ]
}
[report] saved -> c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\enrich_report.json


In [25]:
# 11) Quick Probe: Analyze One Verse by ID
probe_id = "1.1.1"  # change as needed
rec_map = {r["id"]: r for r in records}
rec = rec_map.get(probe_id)
if not rec:
    print(f"[probe] id {probe_id} not found in loaded records")
else:
    out = enrich_record(rec)
    print("id:", out["id"]) 
    print("IAST:", out["iast"][:120])
    print("IPA:", (out.get("ipa") or "")[:120])
    for m in out.get("morph", [])[:10]:
        print("-", m["token"], "=>", (m.get("lemma") or "")[:80])


id: 1.1.1
IAST: oṃ tapaḥ svādhyāyanirataṃ tapasvī vāgvidāṃ varam. nāradaṃ paripapraccha vālmīkirmunipuṅgavam..
IPA: oːm t̪əpəhə̆ s̪ʋɑːd̪ʰjɑːjən̪ɪɾət̪ə̃ t̪əpəs̪ʋiː ʋɑːgʋɪd̪ɑ̃ː ʋəɾəm. n̪ɑːɾəd̪ə̃ pəɾɪpəpɾət͡ʃt͡ʃʰə ʋɑːlmiːkɪɾmun̪ɪpuŋgəʋəm..
- तपः => 
- स्वाध्यायनिरतं => 
- तपस्वी => 
- वाग्विदां => 
- वरम् => 
- नारदं => 
- परिपप्रच्छ => 
- वाल्मीकिर्मुनिपुङ्गवम् => 


In [26]:
# 12) Optional: CLTK Lemma/POS for Latin/Greek (stub)

def analyze_latin_greek(text: str):
    if not cltk:
        print("[cltk] not installed; skip")
        return []
    try:
        # This is a simplified placeholder; full CLTK pipelines require language-specific models
        from cltk.alphabet.text_normalize import normalize_unicode
        norm = normalize_unicode(text)
        # Return dummy structure
        toks = norm.split()
        return [{"token": t, "lemma": t.lower(), "pos": "", "feats": ""} for t in toks]
    except Exception as e:
        print(f"[cltk] error: {e}")
        return []

print("[cltk] stub ready")


[cltk] stub ready


In [27]:
# 13) Optional: Indic-NLP Script Conversion Demo
samples = [r.get("sa_verse", "") for r in records[:3]]
if sanscript is not None:
    try:
        for s in samples:
            dev = s
            # Demonstrate Devanagari -> Bengali
            ben = _itr(dev, sanscript.DEVANAGARI, sanscript.BENGALI)
            print("DEV:", dev[:40], "=> BEN:", ben[:40])
    except Exception as e:
        print("[indic] script conversion failed:", e)
else:
    print("[indic] indic_transliteration not available")


DEV: ॐ तपः स्वाध्यायनिरतं तपस्वी वाग्विदां वर => BEN: ওঁ তপঃ স্বাধ্যাযনিরতং তপস্বী বাগ্বিদাং ব
DEV: कोन्वस्मिन् साम्प्रतं लोके गुणवान् कश्च  => BEN: কোন্বস্মিন্ সাম্প্রতং লোকে গুণবান্ কশ্চ 
DEV: चारित्रेण च को युक्तः सर्वभूतेषु को हितः => BEN: চারিত্রেণ চ কো যুক্তঃ সর্বভূতেষু কো হিতঃ


In [28]:
# 14) Lightweight Tests (assertions) for Core Helpers
try:
    sample = records[0]["sa_verse"]
    iast = dev_to_iast(sample)
    roundtrip = iast_to_dev(iast)
    assert isinstance(iast, str) and len(iast) > 0, "Transliteration to IAST failed"
    assert isinstance(roundtrip, str) and len(roundtrip) > 0, "Transliteration round-trip failed"
    print("[test] transliteration OK")
except AssertionError as e:
    print("[test] transliteration FAIL:", e)

try:
    toks = split_sanskrit(sample)
    assert isinstance(toks, list) and len(toks) > 0, "Tokenization failed"
    print("[test] tokenization OK")
except AssertionError as e:
    print("[test] tokenization FAIL:", e)

try:
    analyses = shp_analyze_token("रामः")
    assert isinstance(analyses, list), "SHP analyze did not return list"
    print("[test] SHP client OK (list)")
except AssertionError as e:
    print("[test] SHP client FAIL:", e)

try:
    ip = to_ipa_dev("रामः")
    assert isinstance(ip, str), "IPA not str"
    print("[test] IPA function OK (type)")
except AssertionError as e:
    print("[test] IPA function FAIL:", e)


[test] transliteration OK
[test] tokenization OK
[time] SHP रामः took 2.207s
[time] SHP रामः took 2.207s
[test] SHP client OK (list)
[test] IPA function OK (type)
[test] SHP client OK (list)
[test] IPA function OK (type)


## 15) Load Classical Corpora: Aeneid (Latin), Iliad (Greek), Vedas (Sanskrit)

Place source texts under `data/external/classical/` using this structure (or adjust patterns in the next cell):
- Latin (Aeneid): `data/external/classical/latin/aeneid/Book1.txt`, `Book2.txt`, ...
- Greek (Iliad): `data/external/classical/greek/iliad/Book1.txt`, ...
- Sanskrit (Vedas): `data/external/classical/sanskrit/veda/<collection>/<file>.txt` (e.g., Rigveda/hymn files).

Notes:
- For JSONL inputs, ensure each line has a `text` field (and optional `book`, `line`).
- This loader normalizes to a simple schema per line: `id`, `work`, `language`, `book`, `line`, `text`.
- Outputs are written to `data/processed/classical_*.jsonl`.


In [29]:
# 15) Loader for Aeneid (Latin), Iliad (Greek), and Vedas (Sanskrit)
from pathlib import Path
import json, re

CLASSICAL_DIR = DATA_DIR / "external" / "classical"
LATIN_DIR = CLASSICAL_DIR / "latin" / "aeneid"
GREEK_DIR = CLASSICAL_DIR / "greek" / "iliad"
VEDA_DIR = CLASSICAL_DIR / "sanskrit" / "veda"

OUT_LATIN = PROCESSED_DIR / "classical_aeneid.jsonl"
OUT_GREEK = PROCESSED_DIR / "classical_iliad.jsonl"
OUT_VEDA = PROCESSED_DIR / "classical_veda.jsonl"

CLASSICAL_DIR.mkdir(parents=True, exist_ok=True)


def iter_lines_from_path(p: Path):
    """Yield (book, line_no, text) from .txt or .jsonl."""
    if p.suffix.lower() == ".jsonl":
        with p.open("r", encoding="utf-8", errors="ignore") as f:
            for i, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    txt = str(obj.get("text", "")).strip()
                    book = str(obj.get("book", p.stem))
                    ln = int(obj.get("line", i))
                    if txt:
                        yield (book, ln, txt)
                except Exception:
                    continue
    else:
        # Assume plain text, one logical line per line, or verses separated by blank lines
        book = p.stem
        with p.open("r", encoding="utf-8", errors="ignore") as f:
            for i, raw in enumerate(f, start=1):
                txt = raw.strip()
                if not txt:
                    continue
                yield (book, i, txt)


def normalize_records(root: Path, work: str, language: str):
    recs = []
    if not root.exists():
        return recs
    paths = sorted(list(root.rglob("*.txt")) + list(root.rglob("*.jsonl")))
    for p in paths:
        for book, line_no, text in iter_lines_from_path(p):
            rid = f"{work}.{book}.{line_no}"
            recs.append({
                "id": rid,
                "work": work,
                "language": language,
                "book": str(book),
                "line": int(line_no),
                "text": text,
            })
    return recs

print("[classical] scanning...")
aeneid = normalize_records(LATIN_DIR, work="aeneid", language="la")
iliad = normalize_records(GREEK_DIR, work="iliad", language="grc")
veda = normalize_records(VEDA_DIR, work="veda", language="sa")

print({"aeneid": len(aeneid), "iliad": len(iliad), "veda": len(veda)})

if aeneid:
    write_jsonl(OUT_LATIN, aeneid)
    print(f"[classical] wrote {len(aeneid)} -> {OUT_LATIN}")
if iliad:
    write_jsonl(OUT_GREEK, iliad)
    print(f"[classical] wrote {len(iliad)} -> {OUT_GREEK}")
if veda:
    write_jsonl(OUT_VEDA, veda)
    print(f"[classical] wrote {len(veda)} -> {OUT_VEDA}")

print("[classical] done. Place files under data/external/classical/* and re-run this cell.")


[classical] scanning...
{'aeneid': 0, 'iliad': 0, 'veda': 0}
[classical] done. Place files under data/external/classical/* and re-run this cell.


## 16) Ingest Pāṇini’s Aṣṭādhyāyī (Sanskrit Wikisource)

Goal: fetch all eight adhyāyas from Sanskrit Wikisource, parse sutra numbering, normalize to JSONL, and save to `data/processed/ashtadhyayi.jsonl`.

Notes:
- Source: Sanskrit Wikisource (CC BY-SA 4.0). We will record `source = "wikisource_sa"`.
- Heuristics: extract lines matching sutra numbers like `1.1.1` (handles Devanagari digits too), capture Sanskrit text, and add IAST via transliteration.
- You can re-run to refresh; output is overwritten but backed up if needed.


In [33]:
# 16) Aṣṭādhyāyī Scraper (Wikisource Sanskrit)
from urllib.parse import urljoin
import re

ASHTA_BASE = "https://sa.wikisource.org/"
ADHYAYA_PATHS = [
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A5%8D%E0%A4%B0%E0%A4%A5%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%A6%E0%A5%8D%E0%A4%B5%E0%A4%BF%E0%A4%A4%E0%A5%80%E0%A4%AF%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%A4%E0%A5%83%E0%A4%A4%E0%A5%80%E0%A4%AF%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%9A%E0%A4%A4%E0%A5%81%E0%A4%B0%E0%A5%8D%E0%A4%A5%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A4%9E%E0%A5%8D%E0%A4%9A%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%B7%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%B8%E0%A4%AA%E0%A5%8D%E0%A4%A4%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
    "wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83",
]

# Devanagari digit map to ASCII
DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")

# Strict sutra id pattern: 1..8 books, 1..4 chapters, 1-3 digit sutra numbers
SUTRA_RE = re.compile(r"\b([1-8]\.[1-4]\.\d{1,3})\b")

OUT_ASHTA = PROCESSED_DIR / "ashtadhyayi.jsonl"


def extract_sutras_from_html(html: str):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "html.parser")
    # Gather all paragraph or list text nodes
    blocks = []
    for sel in ["#mw-content-text p", "#mw-content-text li", "div.mw-parser-output p", "div.mw-parser-output li"]:
        for node in soup.select(sel):
            txt = node.get_text(" ", strip=True)
            if txt:
                blocks.append(txt)
    # Fallback: whole page text
    if not blocks:
        blocks = [soup.get_text(" ", strip=True)]

    recs = []
    for blk in blocks:
        blk_norm = blk.translate(DEV_DIGITS)
        matches = list(SUTRA_RE.finditer(blk_norm))
        if not matches:
            continue
        for i, m in enumerate(matches):
            sid = m.group(1)
            # span for this sutra is between current match end and next match start (or end of block)
            start = m.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(blk_norm)
            segment_norm = blk_norm[start:end]
            # Trim at first danda marker within the segment
            cut = None
            for mark in ["।", "॥"]:
                pos = segment_norm.find(mark)
                if pos != -1:
                    cut = pos
                    break
            if cut is not None:
                end = start + cut
            # Slice original block using same indices (digit normalization is 1:1)
            sa_text = blk[start:end].strip()
            sa_text = re.sub(r"\s+", " ", sa_text)
            book, chapter, sutra = sid.split(".")
            recs.append({
                "id": sid,
                "book": book,
                "chapter": chapter,
                "sutra": sutra,
                "sa": sa_text,
                "iast": dev_to_iast(sa_text) if sa_text else "",
                "source": "wikisource_sa",
            })
    return recs

print("[ashta] fetching...")
all_recs = []
for path in ADHYAYA_PATHS:
    url = urljoin(ASHTA_BASE, path)
    try:
        with Timer(f"fetch {url}"):
            r = requests.get(url, headers={"User-Agent": "nyaya-quantum-nlp/0.1"}, timeout=30)
            r.raise_for_status()
            html = r.text
        recs = extract_sutras_from_html(html)
        print(f"[ashta] {url} -> {len(recs)} candidates")
        all_recs.extend(recs)
        time.sleep(0.5)
    except Exception as e:
        print(f"[ashta] error {url}: {e}")

# Deduplicate by id, prefer longer sa
by_id = {}
for r in all_recs:
    k = r["id"]
    if k not in by_id or len(r.get("sa", "")) > len(by_id[k].get("sa", "")):
        by_id[k] = r

final = [by_id[k] for k in sorted(by_id.keys(), key=lambda x: tuple(map(int, x.split('.'))))]
print(f"[ashta] total unique sutras: {len(final)}")

if final:
    write_jsonl(OUT_ASHTA, final)
    print(f"[ashta] wrote {len(final)} -> {OUT_ASHTA}")
else:
    print("[ashta] no records parsed; inspect page structure or adjust regex")

[ashta] fetching...
[time] fetch https://sa.wikisource.org/wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A5%8D%E0%A4%B0%E0%A4%A5%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83 took 0.337s
[time] fetch https://sa.wikisource.org/wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A5%8D%E0%A4%B0%E0%A4%A5%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83 took 0.337s
[ashta] https://sa.wikisource.org/wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A5%8D%E0%A4%B0%E0%A4%A5%E0%A4%AE%E0%A4%83_%E0%A4%85%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A4%83 -> 702 candidates
[ashta] https://sa.wikisource.org/wiki/%E0%A4%85%E0%A4%B7%E0%A5%8D%E0%A4%9F%E0%A4%BE%E0%A4%A7%E0%A5%8D%E0%A4%AF%E0%A4%BE%E0%A4%AF%E0%A5%80/%E0%A4%AA%E0%A5

## 17) Ingest Aṣṭādhyāyī from GitHub dataset (ashtadhyayi-data)

Primary source: an open dataset on GitHub containing structured sūtras and ancillary lists (Dhātupāṭha, Gaṇapāṭha).

How to use:
- Place or clone the dataset into `data/external/ashtadhyayi-data/` OR provide a ZIP URL in the next cell.
- Run the next cell to normalize into `data/processed/ashtadhyayi.jsonl`.

Schema fields per record:
- id (e.g., `1.1.1`), book, chapter, sutra, sa (Sanskrit), iast (optional), gloss (optional), examples (list), refs (list), source_tags.


In [ ]:
# 17) ashtadhyayi-data loader (local path or ZIP URL)
import csv, io, zipfile, json, re
from urllib.parse import urlparse
import requests as _req

ASHTA_GH_DIR = DATA_DIR / "external" / "ashtadhyayi-data"
ASHTA_ZIP_URL = ""  # e.g., https://github.com/drdhaval2785/ashtadhyayi-data/archive/refs/heads/master.zip
ASHTA_OUT = PROCESSED_DIR / "ashtadhyayi.jsonl"


def _maybe_download_zip(url: str) -> Path | None:
    if not url:
        return None
    tmp_zip = CACHE_DIR / "ashtadhyayi-data.zip"
    with Timer("download ashtadhyayi-data.zip"):
        r = _req.get(url, stream=True, timeout=60)
        r.raise_for_status()
        with tmp_zip.open("wb") as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
    return tmp_zip


def _extract_zip_to_dir(zip_path: Path, target: Path) -> Path:
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target)
    return target


def _iter_dataset_files(root: Path):
    # Yield candidate CSV/JSON under typical folders
    for p in sorted(root.rglob("*.csv")):
        yield p
    for p in sorted(root.rglob("*.json")):
        yield p


def _normalize_sutra_id(val: str) -> str:
    s = str(val or "").strip()
    s = s.translate(str.maketrans("०१२३४५६७८९", "0123456789"))
    # accept formats like 1.1.1 or 1.1.01
    m = re.match(r"^(\d+)\.(\d+)\.(\d+)$", s)
    if m:
        a, b, c = m.groups()
        return f"{int(a)}.{int(b)}.{int(c)}"
    return s


def _record_from_row(row: dict) -> dict:
    # Try to map common field names seen in ashtadhyayi-data variants
    sid = _normalize_sutra_id(row.get("sutra", row.get("id", row.get("sId", ""))))
    sa = row.get("sanskrit", row.get("sutra_text", row.get("sa", "")))
    iast = row.get("iast", "")
    gloss = row.get("meaning", row.get("gloss", ""))
    examples = row.get("examples", [])
    if isinstance(examples, str):
        try:
            examples = json.loads(examples)
        except Exception:
            examples = [examples]
    refs = row.get("refs", row.get("references", []))
    if isinstance(refs, str):
        try:
            refs = json.loads(refs)
        except Exception:
            refs = [refs]
    book = chapter = sutra = ""
    try:
        if sid and sid.count(".") == 2:
            book, chapter, sutra = sid.split(".")
    except Exception:
        pass
    return {
        "id": sid,
        "book": book,
        "chapter": chapter,
        "sutra": sutra,
        "sa": sa,
        "iast": iast or (dev_to_iast(sa) if sa else ""),
        "gloss": gloss,
        "examples": examples if isinstance(examples, list) else [],
        "refs": refs if isinstance(refs, list) else [],
        "source_tags": ["ashtadhyayi-data"],
    }


def load_ashtadhyayi_dataset(local_dir: Path = ASHTA_GH_DIR, zip_url: str = ASHTA_ZIP_URL):
    root = None
    if local_dir.exists():
        root = local_dir
    elif zip_url:
        z = _maybe_download_zip(zip_url)
        if z:
            root = _extract_zip_to_dir(z, CACHE_DIR / "ashtadhyayi-data-extract")
    if root is None or not root.exists():
        print("[ashta-gh] dataset not found; set ASHTA_ZIP_URL or place files under data/external/ashtadhyayi-data/")
        return []

    recs = []
    for p in _iter_dataset_files(root):
        try:
            if p.suffix.lower() == ".csv":
                with p.open("r", encoding="utf-8", errors="ignore") as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        r = _record_from_row(row)
                        if r.get("id"):
                            recs.append(r)
            elif p.suffix.lower() == ".json":
                obj = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
                if isinstance(obj, list):
                    for row in obj:
                        if isinstance(row, dict):
                            r = _record_from_row(row)
                            if r.get("id"):
                                recs.append(r)
        except Exception as e:
            print(f"[ashta-gh] skip {p}: {e}")
    # Dedup by id, prefer richer (longer sa/gloss)
    by_id = {}
    for r in recs:
        k = r.get("id", "")
        if not k:
            continue
        prev = by_id.get(k)
        if not prev:
            by_id[k] = r
        else:
            score = len(r.get("sa", "")) + len(r.get("gloss", ""))
            score_prev = len(prev.get("sa", "")) + len(prev.get("gloss", ""))
            if score > score_prev:
                by_id[k] = r
    out = [by_id[k] for k in sorted(by_id.keys(), key=lambda x: tuple(map(int, x.split('.'))) if x and x.count('.')==2 else (999,999,999))]
    return out

print("[ashta-gh] loading...")
with Timer("ashtadhyayi-data load"):
    gh_recs = load_ashtadhyayi_dataset()
print(f"[ashta-gh] records: {len(gh_recs)}")
if gh_recs:
    write_jsonl(ASHTA_OUT, gh_recs)
    print(f"[ashta-gh] wrote {len(gh_recs)} -> {ASHTA_OUT}")
else:
    print("[ashta-gh] no records; set ASHTA_ZIP_URL or place the dataset locally.")

## 18) Rule Schema Enrichment for Aṣṭādhyāyī

We extend raw sūtra records into a rule-oriented schema usable by derivation/diagram tooling.

Output file: `data/processed/ashtadhyayi_rules.jsonl`

Per-record fields (superset):
- id, book, chapter, sutra, sa, iast, gloss, examples, refs, source_tags
- rule_type: one of {"operational", "meta", "unspecified"}
- labels: keyword tags inferred from text (e.g., sandhi, pratyaya, lakāra)
- conditions: []  (placeholder for left-hand side / triggers)
- outputs: []     (placeholder for right-hand side / results)
- notes: free-form text for curation

These are initial heuristics; manual curation and future parsers will refine conditions/outputs.

In [34]:
# 18) Enrich Aṣṭādhyāyī records -> rule schema
ASHTA_SRC = PROCESSED_DIR / "ashtadhyayi.jsonl"
ASHTA_RULES_OUT = PROCESSED_DIR / "ashtadhyayi_rules.jsonl"

raw = read_jsonl(ASHTA_SRC)
print(f"[rules] loaded {len(raw)} raw records from {ASHTA_SRC}")

KEYWORDS = {
    "sandhi": ["सन्धि", "sandhi"],
    "pratyaya": ["प्रत्यय", "प्रत्यया", "pratyaya"],
    "dhatu": ["धातु", "dhātu"],
    "sup": ["सुप्"],
    "tin": ["तिङ्", "ting"],
    "lakaara": ["लकार", "लकारः", "lakāra"],
}


def infer_labels(text: str):
    s = (text or "").lower()
    labs = []
    for k, kws in KEYWORDS.items():
        for kw in kws:
            if kw.lower() in s:
                labs.append(k)
                break
    return sorted(set(labs))


def guess_rule_type(text: str):
    s = (text or "").lower()
    if any(k in s for k in ["संज्ञा", "saṃjñā", "saMjJA", "definition"]):
        return "meta"
    if any(k in s for k in ["प्रतिषेध", "निषेध", "prohibition"]):
        return "meta"
    # default to operational
    return "operational"


rules = []
for r in raw:
    sa = r.get("sa") or r.get("iast") or ""
    rule = dict(r)
    rule["rule_type"] = guess_rule_type(sa)
    rule["labels"] = infer_labels(sa)
    rule["conditions"] = []
    rule["outputs"] = []
    rule["notes"] = ""
    rules.append(rule)

if rules:
    write_jsonl(ASHTA_RULES_OUT, rules)
    print(f"[rules] wrote {len(rules)} -> {ASHTA_RULES_OUT}")
else:
    print("[rules] nothing to write (no raw records)")


[rules] loaded 3983 raw records from c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\ashtadhyayi.jsonl
[rules] wrote 3983 -> c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\ashtadhyayi_rules.jsonl
[rules] wrote 3983 -> c:\Users\thepe\OneDrive\Desktop\nyaya-quantum-nlp\data\processed\ashtadhyayi_rules.jsonl


## 19) Review Aṣṭādhyāyī Datasets (raw + rules)

This section validates and summarizes the generated datasets:
- Files: `data/processed/ashtadhyayi.jsonl`, `data/processed/ashtadhyayi_rules.jsonl`
- Checks: counts, duplicates, ID sortability, book/chapter distribution
- Rules: rule_type and labels distribution, field completeness
- Samples printed for manual inspection

In [35]:
# 19) Data Review: ashtadhyayi.jsonl and ashtadhyayi_rules.jsonl
from collections import Counter, defaultdict

RAW_PATH = PROCESSED_DIR / "ashtadhyayi.jsonl"
RULES_PATH = PROCESSED_DIR / "ashtadhyayi_rules.jsonl"

raw = read_jsonl(RAW_PATH)
rules = read_jsonl(RULES_PATH)

print("[review] files present:")
print({"raw_exists": RAW_PATH.exists(), "rules_exists": RULES_PATH.exists()})
print(f"[review] raw count: {len(raw)}; rules count: {len(rules)}")

# Helper: numeric id key

def _id_key(sid: str):
    try:
        a, b, c = sid.split('.')
        return (int(a), int(b), int(c))
    except Exception:
        return (999, 999, 999)

# 1) Basic ID diagnostics
raw_ids = [r.get("id", "") for r in raw]
rules_ids = [r.get("id", "") for r in rules]
print("[review] duplicates (raw):", len(raw_ids) - len(set(raw_ids)))
print("[review] duplicates (rules):", len(rules_ids) - len(set(rules_ids)))

sortable_raw = sum(1 for sid in raw_ids if sid.count('.') == 2)
sortable_rules = sum(1 for sid in rules_ids if sid.count('.') == 2)
print(f"[review] sortable IDs — raw: {sortable_raw}/{len(raw_ids)}; rules: {sortable_rules}/{len(rules_ids)}")

# 2) Book/Chapter distribution
bc_raw = Counter((r.get('book', ''), r.get('chapter', '')) for r in raw)
bc_rules = Counter((r.get('book', ''), r.get('chapter', '')) for r in rules)
print("[review] top (book,chapter) raw:", bc_raw.most_common(8))
print("[review] top (book,chapter) rules:", bc_rules.most_common(8))

# 3) Field completeness
missing_raw_sa = sum(1 for r in raw if not (r.get('sa') or r.get('iast')))
missing_rules_labels = sum(1 for r in rules if not r.get('labels'))
missing_rules_type = sum(1 for r in rules if not r.get('rule_type'))
print({
    "missing_raw_text": missing_raw_sa,
    "rules_missing_labels": missing_rules_labels,
    "rules_missing_rule_type": missing_rules_type,
})

# 4) Rule type and labels distribution
rt = Counter(r.get('rule_type','unspecified') for r in rules)
lab_counts = Counter(l for r in rules for l in (r.get('labels') or []))
print("[review] rule_type dist:", dict(rt))
print("[review] labels top:", lab_counts.most_common(15))

# 5) Sample records for manual inspection
print("\n[review] sample raw (first 5 sorted by id):")
for r in sorted(raw, key=lambda x: _id_key(x.get('id','')))[:5]:
    print({k: r.get(k) for k in ["id","book","chapter","sutra","sa","iast"]})

print("\n[review] sample rules (first 5 sorted by id):")
for r in sorted(rules, key=lambda x: _id_key(x.get('id','')))[:5]:
    print({k: r.get(k) for k in ["id","rule_type","labels","gloss"]})

# 6) Sanity check: coverage match
raw_set, rules_set = set(raw_ids), set(rules_ids)
only_in_raw = sorted(list(raw_set - rules_set), key=_id_key)[:10]
only_in_rules = sorted(list(rules_set - raw_set), key=_id_key)[:10]
print({
    "only_in_raw": only_in_raw,
    "only_in_rules": only_in_rules,
})

print("[review] done.")

[review] files present:
{'raw_exists': True, 'rules_exists': True}
[review] raw count: 3983; rules count: 3983
[review] duplicates (raw): 0
[review] duplicates (rules): 0
[review] sortable IDs — raw: 3983/3983; rules: 3983/3983
[review] top (book,chapter) raw: [(('6', '1'), 223), (('6', '2'), 199), (('3', '2'), 188), (('4', '1'), 178), (('3', '3'), 176), (('6', '4'), 175), (('4', '3'), 168), (('5', '4'), 160)]
[review] top (book,chapter) rules: [(('6', '1'), 223), (('6', '2'), 199), (('3', '2'), 188), (('4', '1'), 178), (('3', '3'), 176), (('6', '4'), 175), (('4', '3'), 168), (('5', '4'), 160)]
{'missing_raw_text': 0, 'rules_missing_labels': 3919, 'rules_missing_rule_type': 0}
[review] rule_type dist: {'operational': 3903, 'meta': 80}
[review] labels top: [('pratyaya', 28), ('dhatu', 22), ('sup', 9), ('tin', 9)]

[review] sample raw (first 5 sorted by id):
{'id': '1.1.1', 'book': '1', 'chapter': '1', 'sutra': '1', 'sa': 'वृद्धिरादैच्', 'iast': 'vṛddhirādaic'}
{'id': '1.1.2', 'book': '1

## 20) Finalize and Package Aṣṭādhyāyī Datasets

This section performs the final cleanup and prepares a release package:
- Cleanup demo artifacts (temporary verse enrichments)
- Verify and copy `ashtadhyayi.jsonl` and `ashtadhyayi_rules.jsonl`
- Create a dated release folder and ZIP under `data/releases/`
- Include a small README with source and license notes (Wikisource, CC BY-SA 4.0)

Run this after Sections 16–19 have been executed successfully.

In [ ]:
# 20) Finalization: cleanup + package release
from pathlib import Path
import shutil, zipfile, datetime, json

# Paths
RAW_PATH = PROCESSED_DIR / "ashtadhyayi.jsonl"
RULES_PATH = PROCESSED_DIR / "ashtadhyayi_rules.jsonl"
ENRICH_TMP = PROCESSED_DIR / "gui_verses_enriched.jsonl"  # demo artifact
REPORT_PATH = PROCESSED_DIR / "enrich_report.json"        # demo artifact
RELEASES_DIR = DATA_DIR / "releases"
RELEASES_DIR.mkdir(parents=True, exist_ok=True)

# 1) Cleanup demo artifacts (keep backups under processed/backups already handled earlier)
removed = []
for p in [ENRICH_TMP, REPORT_PATH]:
    if p.exists():
        try:
            p.unlink()
            removed.append(str(p))
        except Exception:
            pass
print({"removed_demo_files": removed})

# 2) Validate outputs exist
if not RAW_PATH.exists() or not RULES_PATH.exists():
    raise SystemExit("[finalize] Required files missing. Run sections 16–19 first.")

# 3) Create release folder with timestamp
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
rel_root = RELEASES_DIR / f"ashtadhyayi_{stamp}"
rel_root.mkdir(parents=True, exist_ok=True)

# 4) Copy core artifacts
raw_dest = rel_root / RAW_PATH.name
rules_dest = rel_root / RULES_PATH.name
shutil.copy2(RAW_PATH, raw_dest)
shutil.copy2(RULES_PATH, rules_dest)

# 5) Write README for release
readme_text = f"""
Aṣṭādhyāyī Dataset (Processed)

Files:
- {raw_dest.name}
- {rules_dest.name}

Source notes:
- Sanskrit Wikisource (CC BY-SA 4.0) for raw text where applicable.
- Optional GitHub ashtadhyayi-data for structured variants.

License and attribution:
- If using Wikisource-derived text, reproduce the CC BY-SA 4.0 license and attribution per page requirements.
- This package is prepared by nyaya-quantum-nlp tooling testbed (Section 16–19).

Build stamp: {stamp}
"""
(rel_root / "README.txt").write_text(readme_text, encoding="utf-8")

# 6) Zip the release folder
zip_path = RELEASES_DIR / f"ashtadhyayi_{stamp}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [raw_dest, rules_dest, rel_root / "README.txt"]:
        zf.write(p, arcname=p.name)

print({
    "release_dir": str(rel_root),
    "zip": str(zip_path),
    "contents": [raw_dest.name, rules_dest.name, "README.txt"],
})
print("[finalize] done.")